# 02 — Frame Labeling

**Primary path (free):** Run the aggregate query (generated dynamically from
`src/dictionaries.py` via `src/build_query.py`) and inspect the frame distributions. No
per-article data needed.

**Secondary path (spot-checking only):** Run the full raw extraction and the Python
preprocessing pipeline to manually verify that the frame labels make sense.

Both query/save cells below overwrite `data/processed/monthly_frames.csv` and
`data/raw/gdelt_genai_gov.csv` unconditionally — the lexicons/queries changed since those
files were last produced (see `MERGE_PLAN.md`), so the previous skip-if-exists guards have
been removed to avoid silently analyzing stale data.

Start with Part A. Only run Part B if you want to audit individual article examples.

### Grounding

Coding articles into a small set of generic frames via keyword dictionaries follows
Semetko & Valkenburg (2000)'s methodological precedent, building on Entman (1993)'s
definition of framing/salience. DiMaggio et al. (2013) is the explicit counterpoint —
topic modeling can surface latent structure a fixed dictionary will miss — cited here as
the tradeoff this project consciously accepts for transparency and reproducibility over
discovery. See `MERGE_PLAN.md`'s grounding map for the full citation-to-design-choice
mapping.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..')))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.analysis import load_agg, monthly_volume_agg, frame_shares_agg
from src.dictionaries import MILESTONES

sns.set_theme(style='whitegrid', palette='colorblind')

---
## Part A — Aggregated path (primary)

Generates the aggregate query dynamically from `src/dictionaries.py` and runs it via
BigQuery, overwriting `data/processed/monthly_frames.csv`.

In [ ]:
import os
from pathlib import Path
from google.cloud import bigquery

from src.build_query import build_aggregate_query

AGG_PATH = Path('../data/processed/monthly_frames.csv')
if AGG_PATH.exists():
    print(f'Overwriting stale {AGG_PATH} (lexicons/queries changed since it was last produced).')

PROJECT = os.environ.get('BIGQUERY_PROJECT', 'genai-gdelt')
client = bigquery.Client(project=PROJECT)
sql = build_aggregate_query()
print('Running aggregate_frames query (generated from src/dictionaries.py) ...')
df_bq = client.query(sql).to_dataframe()
AGG_PATH.parent.mkdir(parents=True, exist_ok=True)
df_bq.to_csv(AGG_PATH, index=False)
print(f'Saved {len(df_bq)} rows -> {AGG_PATH}')

In [ ]:
agg_df = load_agg(AGG_PATH)
print(f'Loaded {len(agg_df)} months, {agg_df["total_articles"].sum():,} total articles')
agg_df.head()

In [ ]:
# Monthly totals
vol = monthly_volume_agg(agg_df)
print('Date range:', vol.index[0], '→', vol.index[-1])
print('Peak month:', vol.idxmax(), '—', vol.max(), 'articles')
vol.tail()

In [ ]:
# Frame hit totals across all months
frame_cols = [c for c in agg_df.columns if c.startswith('frame_')]
frame_totals = agg_df[frame_cols].sum().rename(index=lambda x: x.replace('frame_', ''))

fig, ax = plt.subplots(figsize=(8, 4))
frame_totals.sort_values().plot(kind='barh', ax=ax)
ax.set_title('Total frame keyword matches across all months')
ax.set_xlabel('Number of matching articles')
plt.tight_layout()
plt.show()

print('\nFrame totals:')
print(frame_totals.sort_values(ascending=False))

In [ ]:
# Frame shares over time
shares = frame_shares_agg(agg_df)
shares.head()

In [ ]:
import matplotlib.dates as mdates

fig, ax = plt.subplots(figsize=(12, 4))
dates = shares.index.to_timestamp()
ax.stackplot(
    dates,
    [shares[col] for col in shares.columns],
    labels=[c.replace('_', ' ').title() for c in shares.columns],
    alpha=0.85,
)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45, ha='right')
ax.set_ylabel('Frame share')
ax.set_ylim(0, 1)
ax.set_title('Frame distribution preview (before finalising for paper)')
ax.legend(loc='upper left', fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

### Sanity checks

Look for:
- Any months with suspiciously high or low totals (data gaps)
- Frame shares that seem implausible (e.g. one frame dominating at >80%)
- A visible uptick around Nov 2022 (ChatGPT launch) — if absent, revisit the GenAI filter

In [ ]:
# Months with unusually low article counts (potential data gaps)
mean_vol = vol.mean()
low = vol[vol < mean_vol * 0.3]
if low.empty:
    print('No low-volume months detected.')
else:
    print('Potential data gaps:')
    print(low)

---
## Part B — Raw article spot-check (optional)

Only run this section if you want to audit individual articles by frame label.

**To get the corpus (~3.6 GB CSV):** either
- Run the dry-run cell below to confirm cost, then the BQ extraction cell, **or**
- Use the Google Drive download cell (faster if you have the sharing link).

In [ ]:
# Dry-run: see bytes scanned before committing
import os
from google.cloud import bigquery
from src.build_query import build_extract_query

PROJECT = os.environ.get('BIGQUERY_PROJECT', 'genai-gdelt-499513')
client = bigquery.Client(project=PROJECT)
extract_sql = build_extract_query()

job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
dry = client.query(extract_sql, job_config=job_config)
gb = dry.total_bytes_processed / 1e9
tb = dry.total_bytes_processed / 1e12
print(f'extract_genai_gov query would scan {gb:.1f} GB  ({tb:.2f} TB)')
print(f'Estimated cost beyond free tier: ~${max(0, tb - 1) * 5:.2f}  (at $5/TB)')
print('Status:', 'SAFE (within 1 TB free tier)' if tb < 1 else f'CHARGEABLE — {tb:.2f} TB')

In [ ]:
# Full extraction — only run after confirming cost above. Overwrites unconditionally:
# the lexicons/queries changed since any previous data/raw/gdelt_genai_gov.csv was
# produced, so a stale skip-if-exists guard would silently reuse old data.
RAW_PATH = Path('../data/raw/gdelt_genai_gov.csv')
if RAW_PATH.exists():
    print(f'Overwriting stale {RAW_PATH} (lexicons/queries changed since it was last produced).')

print('Running extract_genai_gov query — this may take several minutes ...')
df_raw = client.query(extract_sql).to_dataframe()
RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
df_raw.to_csv(RAW_PATH, index=False)
print(f'Saved {len(df_raw):,} rows -> {RAW_PATH}')

In [ ]:
# --- Alternative entry point to the cell above: download a pre-extracted corpus from
# Google Drive instead of querying BigQuery yourself. Skip this cell if you already ran
# the extraction above. ---
# 1. Upload gdelt_genai_gov.csv to personal Google Drive
# 2. Right-click → Share → 'Anyone with the link' → Copy link
# 3. Paste the file ID below (the part between /d/ and /view in the URL)
# pip install gdown  (if not already installed)
import gdown
from pathlib import Path

GDRIVE_FILE_ID = "1q2wkv877hodP7gwJTGzrcsXMw7RDthRU"  # ← e.g. '1aBcDeFgHiJkLmNoPqRsTuVwXyZ'
RAW_PATH = Path('../data/raw/gdelt_genai_gov.csv')

if RAW_PATH.exists():
    print(f'Already exists: {RAW_PATH}  ({RAW_PATH.stat().st_size / 1e9:.1f} GB)')
else:
    RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
    url = f'https://drive.google.com/uc?id={GDRIVE_FILE_ID}'
    gdown.download(url, str(RAW_PATH), quiet=False)

In [ ]:
RAW_PATH = Path('../data/raw/gdelt_genai_gov.csv')
if not RAW_PATH.exists():
    print('Raw corpus not found — skip this section.')
else:
    from src.preprocessing import load_raw, run_pipeline
    from src.dictionaries import FRAME_DICTS

    raw_df = load_raw(RAW_PATH)
    df = run_pipeline(raw_df)
    print(f'Loaded {len(df):,} articles after dedup')

`run_pipeline` now also calls `extract_headline_from_url`, which derives a readable
headline from each article's URL slug and backfills it into `Quotations` for rows where
that field was empty/NaN. A large share of GKG records have empty `Quotations`, which
previously starved keyword/frame matching for those rows (flagged by this notebook's own
spot-check below). See `src/preprocessing.py` for the implementation.

In [ ]:
# Save preprocessed corpus for Figure 4 (regional comparison) and Figures 5-6 (tone).
# Overwrites unconditionally — see the note on the extraction cell above.
INTERIM_PATH = Path('../data/interim/gdelt_preprocessed.parquet')
if 'df' in dir():
    if INTERIM_PATH.exists():
        print(f'Overwriting stale {INTERIM_PATH} (lexicons/queries changed since it was last produced).')
    INTERIM_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(INTERIM_PATH, index=False)
    print(f'Saved {len(df):,} rows -> {INTERIM_PATH}')
else:
    print('df not available — run the pipeline cell above first.')

In [ ]:
# Sample 10 articles per dominant frame to verify labels (per MERGE_PLAN.md's
# recommended follow-up: manually sample ~10 articles per frame now that real
# reconciled data exists to sample from).
if 'df' in dir() and 'dominant_frame' in df.columns:
    SPOT_COLS = ['DATE', 'SourceCommonName', 'dominant_frame', 'Quotations']
    for frame_name in FRAME_DICTS:
        n_frame = (df['dominant_frame'] == frame_name).sum()
        if n_frame == 0:
            continue
        sample = df[df['dominant_frame'] == frame_name][SPOT_COLS].sample(
            min(10, n_frame), random_state=42
        )
        print(f'\n=== {frame_name} (n={n_frame:,}) ===')
        for _, row in sample.iterrows():
            print(f'  [{row["DATE"]}] {row["SourceCommonName"]}')
            print(f'  {str(row["Quotations"])[:250]}')